In [21]:
from google.colab import files
uploaded = files. upload()

Saving departments.csv to departments.csv
Saving employees.csv to employees.csv
Saving employees_nested.json to employees_nested.json
Saving sales.csv to sales.csv


In [22]:
!ls

departments.csv  employees.csv	employees_nested.json  sales.csv  sample_data


In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("PySparkLab").getOrCreate()

In [26]:
df_employees = spark.read.csv("employees.csv", header=True, inferSchema=True)
df_employees.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [27]:
df_departments = spark.read.csv("departments.csv", header=True, inferSchema=True)
df_departments.show()

+----------+---------+
|department| location|
+----------+---------+
|        IT|Bangalore|
|        HR|   Mumbai|
|   Finance|    Delhi|
+----------+---------+



In [28]:
df_sales = spark.read.csv("sales.csv", header=True, inferSchema=True)
df_sales.show()

+-------+------+--------+------+
|sale_id|emp_id| product|amount|
+-------+------+--------+------+
|      1|     1|  Laptop| 75000|
|      2|     2|   Mouse|  1500|
|      3|     3|Keyboard|  3000|
|      4|     1| Monitor| 12000|
|      5|     4|  Laptop| 75000|
|      6|     3|   Mouse|  1000|
|      7|     5|Keyboard|  1500|
|      8|     1|  Laptop| 75000|
+-------+------+--------+------+



In [29]:
df_nested = spark.read.json("employees_nested.json", multiLine=True)
df_nested.show()

+--------------------+------+-----+--------------------+
|             address|emp_id| name|              skills|
+--------------------+------+-----+--------------------+
|{Hyderabad, Telan...|     1|Rahul|       [Python, SQL]|
|{Bangalore, Karna...|     2|Sneha|   [HR, Recruitment]|
|{Chennai, Tamil N...|     3|Arjun|[PySpark, Azure, ...|
+--------------------+------+-----+--------------------+



In [30]:
print("Employees Schema:")
df_employees.printSchema()
print("Departments Schema:")
df_departments.printSchema()
print("Sales Schema:")
df_sales.printSchema()
print("Nested JSON Schema:")
df_nested.printSchema()

Employees Schema:
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- joining_date: date (nullable = true)

Departments Schema:
root
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)

Sales Schema:
root
 |-- sale_id: integer (nullable = true)
 |-- emp_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)

Nested JSON Schema:
root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [31]:
df_employees.show(5)
df_employees.show(3)

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
+------+-----+----------+------+------------+
only showing top 5 rows
+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
+------+-----+----------+------+------------+
only showing top 3 rows


In [32]:
print(f"Total Employees: {df_employees.count()}")

Total Employees: 7


In [33]:
print(df_employees.columns)

['emp_id', 'name', 'department', 'salary', 'joining_date']


In [34]:
df_employees.select("name", "salary").show()

+-----+------+
| name|salary|
+-----+------+
|Rahul| 70000|
|Sneha| 60000|
|Arjun| 75000|
|Priya| 80000|
|Karan| 50000|
|Meera| 72000|
| Amit| 58000|
+-----+------+



In [35]:
df_employees = df_employees.withColumnRenamed("salary", "emp_salary")
df_employees.show()

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2023-01-10|
|     2|Sneha|        HR|     60000|  2022-11-15|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     4|Priya|   Finance|     80000|  2022-12-20|
|     5|Karan|        IT|     50000|  2023-02-05|
|     6|Meera|      NULL|     72000|  2023-04-10|
|     7| Amit|        HR|     58000|  2023-01-18|
+------+-----+----------+----------+------------+



In [36]:
df_employees.filter(col("joining_date") > "2023-01-01").show()

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2023-01-10|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     5|Karan|        IT|     50000|  2023-02-05|
|     6|Meera|      NULL|     72000|  2023-04-10|
|     7| Amit|        HR|     58000|  2023-01-18|
+------+-----+----------+----------+------------+



In [37]:

df_employees.filter(col("emp_salary") > 65000).show()

df_employees.filter(col("department") == "IT").show()
df_employees.filter((col("department") == "IT") & (col("emp_salary") > 70000)).show()

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2023-01-10|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     4|Priya|   Finance|     80000|  2022-12-20|
|     6|Meera|      NULL|     72000|  2023-04-10|
+------+-----+----------+----------+------------+

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2023-01-10|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     5|Karan|        IT|     50000|  2023-02-05|
+------+-----+----------+----------+------------+

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     3|Arjun|        IT|     75000|  2023-03-01|
+------+-----+----------+----------+------------

In [38]:
df_temp = df_employees.drop("joining_date")
df_temp.show()

+------+-----+----------+----------+
|emp_id| name|department|emp_salary|
+------+-----+----------+----------+
|     1|Rahul|        IT|     70000|
|     2|Sneha|        HR|     60000|
|     3|Arjun|        IT|     75000|
|     4|Priya|   Finance|     80000|
|     5|Karan|        IT|     50000|
|     6|Meera|      NULL|     72000|
|     7| Amit|        HR|     58000|
+------+-----+----------+----------+



In [39]:
df_employees.orderBy("emp_salary").show() # Ascending
df_employees.orderBy(col("emp_salary").desc()).show() # Descending

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     5|Karan|        IT|     50000|  2023-02-05|
|     7| Amit|        HR|     58000|  2023-01-18|
|     2|Sneha|        HR|     60000|  2022-11-15|
|     1|Rahul|        IT|     70000|  2023-01-10|
|     6|Meera|      NULL|     72000|  2023-04-10|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     4|Priya|   Finance|     80000|  2022-12-20|
+------+-----+----------+----------+------------+

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     4|Priya|   Finance|     80000|  2022-12-20|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     6|Meera|      NULL|     72000|  2023-04-10|
|     1|Rahul|        IT|     70000|  2023-01-10|
|     2|Sneha|        HR|     60000|  2022-11-15|
|     7| Amit|        HR|     58000|  2023-01-18|

In [40]:
df_employees.orderBy(col("emp_salary").desc()).limit(3).show()

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     4|Priya|   Finance|     80000|  2022-12-20|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     6|Meera|      NULL|     72000|  2023-04-10|
+------+-----+----------+----------+------------+



In [41]:
df_employees.select(
    sum("emp_salary").alias("Total_Salary"),
    avg("emp_salary").alias("Avg_Salary"),
    max("emp_salary").alias("Max_Salary"),
    min("emp_salary").alias("Min_Salary")
).show()

+------------+-----------------+----------+----------+
|Total_Salary|       Avg_Salary|Max_Salary|Min_Salary|
+------------+-----------------+----------+----------+
|      465000|66428.57142857143|     80000|     50000|
+------------+-----------------+----------+----------+



In [42]:
df_employees.groupBy("department").agg(
    count("*").alias("Employee_Count"),
    avg("emp_salary").alias("Avg_Dept_Salary"),
    sum("emp_salary").alias("Total_Dept_Salary")
).show()

+----------+--------------+---------------+-----------------+
|department|Employee_Count|Avg_Dept_Salary|Total_Dept_Salary|
+----------+--------------+---------------+-----------------+
|        HR|             2|        59000.0|           118000|
|      NULL|             1|        72000.0|            72000|
|   Finance|             1|        80000.0|            80000|
|        IT|             3|        65000.0|           195000|
+----------+--------------+---------------+-----------------+



In [43]:
df_employees.join(df_departments, "department", "inner").show()


df_employees.join(df_departments, "department", "left").show()

+----------+------+-----+----------+------------+---------+
|department|emp_id| name|emp_salary|joining_date| location|
+----------+------+-----+----------+------------+---------+
|        IT|     1|Rahul|     70000|  2023-01-10|Bangalore|
|        HR|     2|Sneha|     60000|  2022-11-15|   Mumbai|
|        IT|     3|Arjun|     75000|  2023-03-01|Bangalore|
|   Finance|     4|Priya|     80000|  2022-12-20|    Delhi|
|        IT|     5|Karan|     50000|  2023-02-05|Bangalore|
|        HR|     7| Amit|     58000|  2023-01-18|   Mumbai|
+----------+------+-----+----------+------------+---------+

+----------+------+-----+----------+------------+---------+
|department|emp_id| name|emp_salary|joining_date| location|
+----------+------+-----+----------+------------+---------+
|        IT|     1|Rahul|     70000|  2023-01-10|Bangalore|
|        HR|     2|Sneha|     60000|  2022-11-15|   Mumbai|
|        IT|     3|Arjun|     75000|  2023-03-01|Bangalore|
|   Finance|     4|Priya|     80000|  2

In [44]:
df_employees.join(df_departments, "department", "inner").select("name", "location").show()

+-----+---------+
| name| location|
+-----+---------+
|Rahul|Bangalore|
|Sneha|   Mumbai|
|Arjun|Bangalore|
|Priya|    Delhi|
|Karan|Bangalore|
| Amit|   Mumbai|
+-----+---------+



In [45]:
df_employees = df_employees.withColumn("bonus", col("emp_salary") * 0.10) \
                           .withColumn("company", lit("BotCampus"))
df_employees.show()

+------+-----+----------+----------+------------+------+---------+
|emp_id| name|department|emp_salary|joining_date| bonus|  company|
+------+-----+----------+----------+------------+------+---------+
|     1|Rahul|        IT|     70000|  2023-01-10|7000.0|BotCampus|
|     2|Sneha|        HR|     60000|  2022-11-15|6000.0|BotCampus|
|     3|Arjun|        IT|     75000|  2023-03-01|7500.0|BotCampus|
|     4|Priya|   Finance|     80000|  2022-12-20|8000.0|BotCampus|
|     5|Karan|        IT|     50000|  2023-02-05|5000.0|BotCampus|
|     6|Meera|      NULL|     72000|  2023-04-10|7200.0|BotCampus|
|     7| Amit|        HR|     58000|  2023-01-18|5800.0|BotCampus|
+------+-----+----------+----------+------------+------+---------+



In [46]:
df_employees.filter(col("department").isNull()).show()

df_employees = df_employees.fillna({"department": "Unknown"})

df_clean = df_employees.na.drop()
df_employees.show()

+------+-----+----------+----------+------------+------+---------+
|emp_id| name|department|emp_salary|joining_date| bonus|  company|
+------+-----+----------+----------+------------+------+---------+
|     6|Meera|      NULL|     72000|  2023-04-10|7200.0|BotCampus|
+------+-----+----------+----------+------------+------+---------+

+------+-----+----------+----------+------------+------+---------+
|emp_id| name|department|emp_salary|joining_date| bonus|  company|
+------+-----+----------+----------+------------+------+---------+
|     1|Rahul|        IT|     70000|  2023-01-10|7000.0|BotCampus|
|     2|Sneha|        HR|     60000|  2022-11-15|6000.0|BotCampus|
|     3|Arjun|        IT|     75000|  2023-03-01|7500.0|BotCampus|
|     4|Priya|   Finance|     80000|  2022-12-20|8000.0|BotCampus|
|     5|Karan|        IT|     50000|  2023-02-05|5000.0|BotCampus|
|     6|Meera|   Unknown|     72000|  2023-04-10|7200.0|BotCampus|
|     7| Amit|        HR|     58000|  2023-01-18|5800.0|BotCa